In [23]:
%pip install opencv-python
#Run if you don't have opencv installed.

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
     - -------------------------------------- 0.8/15.8 MB 11.1 MB/s eta 0:00:02
     --------- ------------------------------ 3.9/15.8 MB 13.6 MB/s eta 0:00:01
     ---------------- ----------------------- 6.6/15.8 MB 13.4 MB/s eta 0:00:01
     ----------------------- ---------------- 9.4/15.8 MB 13.0 MB/s eta 0:00:01
     -------------------------- ------------ 10.7/15.8 MB 11.7 MB/s eta 0:00:01
     ------------------------------ -------- 12.3/15.8 MB 10.9 MB/s eta 0:00:01
     ------------------------------------ -- 14.7/15.8 MB 10.8 MB/s eta 0:00:01
     ---------------------------------------- 15.8/15.8 MB 10.6 MB/s  0:00:01
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependenci

  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [21 lines of output]
      + c:\Users\hgao1\OneDrive\Desktop\projects\swhw\eece4632\.venv\Scripts\python.exe C:\Users\hgao1\AppData\Local\Temp\pip-install-h44a6_t4\numpy_a8683c6f0c3b4347860026b628b6f774\vendored-meson\meson\meson.py setup C:\Users\hgao1\AppData\Local\Temp\pip-install-h44a6_t4\numpy_a8683c6f0c3b4347860026b628b6f774 C:\Users\hgao1\AppData\Local\Temp\pip-install-h44a6_t4\numpy_a8683c6f0c3b4347860026b628b6f774\.mesonpy-7z7dji5i -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\hgao1\AppData\Local\Temp\pip-install-h44a6_t4\numpy_a8683c6f0c3b4347860026b628b6f774\.mesonpy-7z7dji5i\meson-python-native-file.ini
      The Meson build system
      Version: 1.2.99
      Source dir: C:\Users\hgao1\AppData\Local\Temp\pip-install-h44a6_t4\numpy_a8683c6f0c3b4347860026b628b6f774
      Build dir: C:\Users\hgao1\AppData\Local\Temp\p

In [ ]:
import cv2
import numpy as np

In [32]:
# can change based on what color we want
COLOR = [
    (np.array([0,   120,  70]),  np.array([10,  255, 255])),  # lower red
    (np.array([170, 120,  70]),  np.array([180, 255, 255])),  # upper red
]

MIN_AREA = 800  # remove small items      
BLUR = (11, 11)  # blur

In [33]:
def mask(hsv: np.ndarray) -> np.ndarray:
    lower = cv2.inRange(hsv, COLOR[0][0], COLOR[0][1])
    upper = cv2.inRange(hsv, COLOR[1][0], COLOR[1][1])
    mask  = cv2.bitwise_or(lower, upper)
 
    # clean up mask
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return mask

In [34]:
def draw(frame: np.ndarray, mask: np.ndarray) -> np.ndarray:
    output = frame.copy()
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    found = 0

    for contour in contours:
        area = cv2.contourArea(contour)
        if area < MIN_AREA: continue
        found += 1
 
        x, y, w, h = cv2.boundingRect(contour)
        cv2.rectangle(output, (x, y), (x + w, y + h), (0, 0, 255), 2)
    return output

In [37]:
def detect_webcam(cam_index: int = 0):
    cap = cv2.VideoCapture(cam_index)
    if not cap.isOpened(): raise RuntimeError("no cam")
 
    while True:
        r, frame = cap.read()
        if not r: break
 
        hsv = cv2.cvtColor(cv2.GaussianBlur(frame, BLUR, 0), cv2.COLOR_BGR2HSV)
        m = mask(hsv)
        output = draw(frame, m)
 
        cv2.imshow("Detection", output)
        cv2.imshow("Mask", m)
 
        if cv2.waitKey(1) & 0xFF == ord("q"): break
 
    cap.release()
    cv2.destroyAllWindows()

In [41]:
detect_webcam()